In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()                                   # fp32 reduction
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * rms).to(dtype) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=4, multiple_of=64):
        super().__init__()
        hidden = int(dim * mult * 2 / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden, bias=False)    # gate branch
        self.w2 = nn.Linear(dim, hidden, bias=False)    # value branch
        self.w3 = nn.Linear(hidden, dim, bias=False)    # project back

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# ---------------- tests ----------------

torch.manual_seed(42)
B, T, D = 4, 32, 768
x = torch.randn(B, T, D)

norm = RMSNorm(D)
y = norm(x)
rms = y.pow(2).mean(-1).sqrt()
assert y.shape == x.shape
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)
print(f"RMSNorm   ok | out RMS {rms.mean():.6f} | params {sum(p.numel() for p in norm.parameters()):,}")

# fp16 stability: large activations must not overflow
xb = torch.randn(2, 8, D) * 300
assert torch.isfinite(norm(xb.half() if torch.cuda.is_available() else xb)).all()
print("RMSNorm   fp16 overflow test ok")

mlp = SwiGLU(D)
z = mlp(x)
assert z.shape == x.shape
print(f"SwiGLU    ok | hidden {mlp.w1.out_features} | params {sum(p.numel() for p in mlp.parameters()):,}")

RMSNorm   ok | out RMS 1.000000 | params 768
RMSNorm   fp16 overflow test ok
SwiGLU    ok | hidden 2048 | params 4,718,592


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RoPE(nn.Module):
    """Contiguous-half rotary embeddings.
    x = [x1 | x2] -> [x1*cos - x2*sin, x2*cos + x1*sin]
    This convention MUST match the inference path exactly."""

    def __init__(self, head_dim, max_seq):
        super().__init__()
        assert head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.head_dim, self.max_seq = head_dim, max_seq

        inv_freq = 1.0 / (10000 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        """x: (B, H, T, head_dim). pos0 is the absolute position of x[..., 0, :],
        used by cached generation where T == 1."""
        dtype = x.dtype
        B, H, T, D = x.shape
        assert D == self.head_dim
        assert pos0 + T <= self.max_seq, f"position {pos0 + T} exceeds max_seq {self.max_seq}"

        half = D // 2
        x = x.float()
        x1, x2 = x[..., :half], x[..., half:]

        cos = self.cos[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)

        return torch.cat([x1 * cos - x2 * sin,
                          x2 * cos + x1 * sin], dim=-1).to(dtype)


class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, rope=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope     = RoPE(self.head_dim, max_seq) if rope else None

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)   # (B, H, T, hd)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)            # never v

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


# ============================================================
# TESTS
# ============================================================

torch.manual_seed(42)
B, T, D, H = 2, 16, 768, 12

attn = CausalSelfAttention(D, H, max_seq=2048, rope=False)
x = torch.randn(B, T, D)
y = attn(x)
assert y.shape == x.shape and torch.isfinite(y).all()
print("attention  shape/finite ok")

# causality: perturbing position t must not affect positions < t
t = 7
x1 = torch.randn(B, T, D)
x2 = x1.clone()
x2[:, t, :] = torch.randn(B, D)
with torch.no_grad():
    y1, y2 = attn(x1), attn(x2)

assert torch.allclose(y1[:, :t], y2[:, :t], atol=1e-6), "LEAK: past positions changed"
assert not torch.allclose(y1[:, t:], y2[:, t:], atol=1e-6), "position t had no effect"
print("attention  causality ok")

# --- RoPE ---
hd, max_seq = 64, 2048
rope = RoPE(hd, max_seq)

q = torch.randn(2, 8, 32, hd)
assert torch.allclose(q.norm(dim=-1), rope(q).norm(dim=-1), atol=1e-5), "not a rotation"
print("rope       norm preservation ok")

# relative position: score depends only on (i - j), with DISTINCT q and k
qv, kv = torch.randn(hd), torch.randn(hd)
qt, kt = torch.zeros(1, 1, 16, hd), torch.zeros(1, 1, 16, hd)
qt[0, 0, 5], kt[0, 0, 3]   = qv, kv      # offset 2
qt[0, 0, 12], kt[0, 0, 10] = qv, kv      # offset 2
qr, kr = rope(qt), rope(kt)

d1 = (qr[0, 0, 5]  * kr[0, 0, 3]).sum()
d2 = (qr[0, 0, 12] * kr[0, 0, 10]).sum()
d3 = (qr[0, 0, 12] * kr[0, 0, 3]).sum()   # offset 9 — must differ

assert torch.allclose(d1, d2, atol=1e-5), f"not relative: {d1:.6f} vs {d2:.6f}"
assert not torch.allclose(d1, d3, atol=1e-3), "position has no effect at all"
print(f"rope       relative position ok  (off2 {d1:.4f}={d2:.4f}, off9 {d3:.4f})")

# pos0 offset must equal the same absolute position computed in one pass
full = rope(qt)
piece = rope(qt[:, :, 12:13], pos0=12)
assert torch.allclose(full[:, :, 12:13], piece, atol=1e-5), "pos0 offset broken"
print("rope       pos0 offset ok")

# --- combined ---
rattn = CausalSelfAttention(768, 12, max_seq=2048)
xr = torch.randn(2, 128, 768)
with torch.no_grad():
    yr = rattn(xr)
assert yr.shape == xr.shape and torch.isfinite(yr).all()
print(f"rope-attn  ok | params {sum(p.numel() for p in rattn.parameters()):,}")

attention  shape/finite ok
attention  causality ok
rope       norm preservation ok
rope       relative position ok  (off2 -0.3179=-0.3179, off9 -4.3299)
rope       pos0 offset ok
rope-attn  ok | params 2,359,296


In [3]:
import os, glob, json
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

DATA = Path("/kaggle/input/datasets/megs132/capstone-1")

for p in sorted(DATA.iterdir()):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

WORK     = "/kaggle/working"
CKPT_DIR = f"{WORK}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_BIN, VAL_BIN = DATA / "bpe32k_train.bin", DATA / "bpe32k_val.bin"
META_PATH, TOK_PATH = DATA / "bpe32k_meta.json", DATA / "bpe32k.json"
TEST_PATH = DATA / "test.json"
VAL_PROMPTS_PATH, VAL_REFS_PATH = DATA / "val_prompts.json", DATA / "val_refs.json"

for p in (TRAIN_BIN, VAL_BIN, META_PATH, TOK_PATH, TEST_PATH,
          VAL_PROMPTS_PATH, VAL_REFS_PATH):
    assert p.exists(), f"missing: {p}"

META = json.load(open(META_PATH))
VOCAB_SIZE = META["vocab_size"]
PAD_ID = META["special_token_ids"]["pad"]
EOS_ID = META["special_token_ids"]["eos"]

tok = Tokenizer.from_file(str(TOK_PATH))
train_bin = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_bin   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

test_data   = json.load(open(TEST_PATH))
VAL_PROMPTS = json.load(open(VAL_PROMPTS_PATH))
VAL_REFS    = json.load(open(VAL_REFS_PATH))

print(f"\nvocab {VOCAB_SIZE:,} | pad {PAD_ID} | eos {EOS_ID}")
print(f"train {len(train_bin):,} tokens | val {len(val_bin):,} tokens")
print(f"test {len(test_data)} | val prompts {len(VAL_PROMPTS)}")
print("\nsample:", tok.decode(train_bin[:60].astype(np.int64).tolist())[:300])

      0.0 MB  __huggingface_repos__.json
      2.3 MB  bpe32k.json
      0.0 MB  bpe32k_meta.json
    434.4 MB  bpe32k_train.bin
      5.1 MB  bpe32k_val.bin
      0.0 MB  corpus_stats.json
      3.0 MB  test.json
    181.6 MB  train_acts.jsonl
    367.1 MB  train_high_courts.jsonl
    296.2 MB  train_supreme_court.jsonl
    183.4 MB  train_tribunals.jsonl
      3.1 MB  val_acts.jsonl
      3.1 MB  val_high_courts.jsonl
      0.9 MB  val_prompts.json
      0.2 MB  val_refs.json
      3.0 MB  val_supreme_court.jsonl
      3.1 MB  val_tribunals.jsonl

vocab 32,000 | pad 0 | eos 3
train 217,201,090 tokens | val 2,560,200 tokens
test 1000 | val prompts 300

sample: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu


In [4]:
class GeneralizedAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, causal=True, cross=False):
        super().__init__()
        assert dim % n_heads == 0

        self.n_heads, self.head_dim = n_heads, dim // n_heads
        self.causal, self.cross = causal, cross

        # separate projections: cross-attn draws q from x, k/v from context
        self.q_proj   = nn.Linear(dim, dim, bias=False)
        self.k_proj   = nn.Linear(dim, dim, bias=False)
        self.v_proj   = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

        # RoPE applies to self-attention only; encoder output is not a
        # sequence the decoder indexes by relative position
        self.rope = None if cross else RoPE(self.head_dim, max_seq)

    def forward(self, x, context=None):
        B, T, C = x.shape
        src = context if self.cross else x
        if self.cross:
            assert context is not None, "cross-attention needs context"
        S = src.shape[1]

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(src).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(src).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)

        out = F.scaled_dot_product_attention(
            q, k, v, is_causal=(self.causal and not self.cross))
        return self.out_proj(out.transpose(1, 2).reshape(B, T, C))

In [5]:
# ============================================================
# 2. Encoder block
# ============================================================

class EncoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.attn = GeneralizedAttention(
            dim=dim,
            n_heads=n_heads,
            max_seq=max_seq,
            causal=False,
            cross=False,
        )

        self.norm2 = RMSNorm(dim)

        self.mlp = SwiGLU(dim)

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.mlp(
            self.norm2(x)
        )

        return x


# ============================================================
# 3. Decoder block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()

        # Causal self-attention
        self.norm1 = RMSNorm(dim)

        self.self_attn = GeneralizedAttention(
            dim=dim,
            n_heads=n_heads,
            max_seq=max_seq,
            causal=True,
            cross=False,
        )

        # Cross-attention to encoder
        self.norm2 = RMSNorm(dim)

        self.cross_attn = GeneralizedAttention(
            dim=dim,
            n_heads=n_heads,
            max_seq=max_seq,
            causal=False,
            cross=True,
        )

        # MLP
        self.norm3 = RMSNorm(dim)

        self.mlp = SwiGLU(dim)

    def forward(self, x, encoder_output):
        x = x + self.self_attn(
            self.norm1(x)
        )

        x = x + self.cross_attn(
            self.norm2(x),
            context=encoder_output
        )

        x = x + self.mlp(
            self.norm3(x)
        )

        return x

In [8]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DIM, N_HEADS, MAX_SEQ = 768, 12, 2048
BLOCK = 1024
print(f"device {DEVICE} | vocab {VOCAB_SIZE:,} | dim {DIM} | heads {N_HEADS}")

device cuda | vocab 32,000 | dim 768 | heads 12


In [9]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

USE_AMP = DEVICE == "cuda"

B_SPLIT, B_ENC_LAYERS, B_DEC_LAYERS = 768, 6, 6


class EncoderDecoderModel(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, dim=DIM,
                 n_encoder_layers=6, n_decoder_layers=6,
                 n_heads=N_HEADS, max_seq=MAX_SEQ):
        super().__init__()
        self.vocab_size, self.dim, self.max_seq = vocab_size, dim, max_seq

        self.tok_emb = nn.Embedding(vocab_size, dim)

        self.encoder_blocks = nn.ModuleList(
            [EncoderBlock(dim, n_heads, max_seq) for _ in range(n_encoder_layers)])
        self.norm_e = RMSNorm(dim)

        self.decoder_blocks = nn.ModuleList(
            [DecoderBlock(dim, n_heads, max_seq) for _ in range(n_decoder_layers)])
        self.norm_f = RMSNorm(dim)

        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # tie

        self.apply(self._init_weights)

        std_e = 0.02 / math.sqrt(2 * n_encoder_layers)
        std_d = 0.02 / math.sqrt(2 * n_decoder_layers)

        for b in self.encoder_blocks:
            nn.init.normal_(b.attn.out_proj.weight, 0.0, std_e)
            nn.init.normal_(b.mlp.w3.weight,        0.0, std_e)

        for b in self.decoder_blocks:
            nn.init.normal_(b.self_attn.out_proj.weight,  0.0, std_d)
            nn.init.normal_(b.cross_attn.out_proj.weight, 0.0, std_d)
            nn.init.normal_(b.mlp.w3.weight,              0.0, std_d)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, split=None):
        T = idx.size(1)
        if T > self.max_seq:
            raise ValueError(f"T={T} exceeds max_seq={self.max_seq}")
        split = split or T // 2
        if not 1 <= split <= T:
            raise ValueError(f"invalid split={split} for T={T}")

        e = self.tok_emb(idx[:, :split])
        for b in self.encoder_blocks:
            e = b(e)
        e = self.norm_e(e)                      # normalise once, before cross-attn

        d = self.tok_emb(idx[:, split - 1:])    # start one early -> first pred is at `split`
        for b in self.decoder_blocks:
            d = b(d, e)
        logits = self.head(self.norm_f(d))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size),
                                   targets[:, split - 1:].reshape(-1))
        return logits, loss


def count_unique_params(m):
    return sum(p.numel() for p in {id(p): p for p in m.parameters()}.values())





In [10]:
import glob, torch
from pathlib import Path

hits = glob.glob("/kaggle/input/**/B_encdec.pt", recursive=True)
assert hits, "phase1-model-b not attached"
CKPT_B = Path(hits[0])

model_b = EncoderDecoderModel(
    vocab_size=VOCAB_SIZE, dim=DIM,
    n_encoder_layers=6, n_decoder_layers=6,
    n_heads=N_HEADS, max_seq=MAX_SEQ).to(DEVICE)

sd = torch.load(CKPT_B, map_location="cpu")
model_b.load_state_dict({k: v.float() for k, v in sd.items()}, strict=True)
model_b.eval()
print(f"loaded {count_unique_params(model_b)/1e6:.2f}M params from {CKPT_B.name}")

loaded 123.69M params from B_encdec.pt


In [11]:
import torch
import torch.nn.functional as F


def cached_self_attn(attn, x, cache=None, pos0=0):
    """Decoder causal self-attention with growing KV cache."""
    B, T, C = x.shape

    q = attn.q_proj(x).view(B, T, attn.n_heads, attn.head_dim).transpose(1, 2)
    k = attn.k_proj(x).view(B, T, attn.n_heads, attn.head_dim).transpose(1, 2)
    v = attn.v_proj(x).view(B, T, attn.n_heads, attn.head_dim).transpose(1, 2)

    q, k = attn.rope(q, pos0=pos0), attn.rope(k, pos0=pos0)

    if cache is not None:
        k = torch.cat([cache[0], k], dim=2)
        v = torch.cat([cache[1], v], dim=2)

    out = F.scaled_dot_product_attention(q, k, v, is_causal=(T > 1))
    return attn.out_proj(out.transpose(1, 2).reshape(B, T, C)), (k, v)


def cross_attn_cached(attn, x, kv):
    """Cross-attention against precomputed encoder K/V. No RoPE, no mask."""
    B, T, C = x.shape
    q = attn.q_proj(x).view(B, T, attn.n_heads, attn.head_dim).transpose(1, 2)
    out = F.scaled_dot_product_attention(q, kv[0], kv[1], is_causal=False)
    return attn.out_proj(out.transpose(1, 2).reshape(B, T, C))


@torch.no_grad()
def encode_prompt(model, enc_tokens):
    """Run the encoder once; project its output to per-layer cross K/V once."""
    x = model.tok_emb(enc_tokens)
    for b in model.encoder_blocks:
        x = b(x)
    e = model.norm_e(x)

    cross_kv = []
    for b in model.decoder_blocks:
        a = b.cross_attn
        B, S, _ = e.shape
        k = a.k_proj(e).view(B, S, a.n_heads, a.head_dim).transpose(1, 2)
        v = a.v_proj(e).view(B, S, a.n_heads, a.head_dim).transpose(1, 2)
        cross_kv.append((k, v))
    return e, cross_kv


@torch.no_grad()
def decoder_forward(model, tokens, self_caches, cross_kv, pos0, last_only=True):
    x = model.tok_emb(tokens)
    new_caches = []

    for i, b in enumerate(model.decoder_blocks):
        a, c = cached_self_attn(b.self_attn, b.norm1(x),
                                None if self_caches is None else self_caches[i], pos0)
        x = x + a
        x = x + cross_attn_cached(b.cross_attn, b.norm2(x), cross_kv[i])
        x = x + b.mlp(b.norm3(x))
        new_caches.append(c)

    x = model.norm_f(x)
    return model.head(x[:, -1:] if last_only else x), new_caches


@torch.no_grad()
def prefill_b(model, prompt, split):
    """Encoder on prompt[:, :split]; decoder primed on prompt[:, split-1:],
    matching the training-time convention exactly."""
    _, cross_kv = encode_prompt(model, prompt[:, :split])
    dec = prompt[:, split - 1:]
    logits, caches = decoder_forward(model, dec, None, cross_kv, pos0=0)
    return logits, caches, cross_kv, dec.shape[1]


@torch.no_grad()
def step_b(model, token, self_caches, cross_kv):
    pos0 = self_caches[0][0].shape[2]
    return decoder_forward(model, token, self_caches, cross_kv, pos0=pos0)


# ============================================================
# EQUIVALENCE TEST
# ============================================================

model_b.eval()
torch.manual_seed(42)

T_TEST, SPLIT, N_STEPS = 1024, 768, 4
PRIME = 20                       # decoder positions to prefill before stepping

x = torch.randint(0, VOCAB_SIZE, (1, T_TEST), device=DEVICE)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    full, _ = model_b(x, split=SPLIT)
    print(f"decoder positions in full forward: {full.shape[1]}")

    # prefill on the first PRIME decoder tokens -> its logit is decoder index PRIME-1
    prompt_part = x[:, :SPLIT - 1 + PRIME]
    pl, caches, ckv, dlen = prefill_b(model_b, prompt_part, split=SPLIT)
    assert dlen == PRIME and caches[0][0].shape[2] == PRIME

    got = [pl]
    dec_tokens = x[:, SPLIT - 1:]
    for j in range(PRIME, PRIME + N_STEPS):
        lg, caches = step_b(model_b, dec_tokens[:, j:j + 1], caches, ckv)
        assert caches[0][0].shape[2] == j + 1, "cache length drifted"
        got.append(lg)
    got = torch.cat(got, dim=1)

ref = full[:, PRIME - 1: PRIME - 1 + N_STEPS + 1]
assert got.shape == ref.shape, f"{got.shape} vs {ref.shape}"

diff  = (got.float() - ref.float()).abs()
agree = (got.argmax(-1) == ref.argmax(-1)).float().mean().item() * 100

print(f"\npositions compared : decoder {PRIME-1} .. {PRIME-1+N_STEPS}")
print(f"max abs diff       : {diff.max().item():.6f}")
print(f"per-position max   : {[round(d, 4) for d in diff.amax(-1)[0].tolist()]}")
print(f"top-1 agreement    : {agree:.1f}%")
print("\nPASS" if diff.max().item() < 0.05 and agree == 100.0 else "\nFAIL")

decoder positions in full forward: 257

positions compared : decoder 19 .. 23
max abs diff       : 0.007812
per-position max   : [0.0078, 0.0039, 0.0039, 0.0059, 0.0039]
top-1 agreement    : 100.0%

PASS


In [14]:
import torch

PAD_ID = int(tok.token_to_id("<|pad|>"))


@torch.no_grad()
def generate_batched_b(test_data, model, split_k=16, batch_size=32,
                       max_new_tokens=350, target_words=128, check_every=16,
                       temperature=0.7, top_p=0.9, top_k=50,
                       repetition_penalty=1.1, repetition_window=128,
                       no_repeat_ngram_size=None, verbose=True):
    """Encoder reads prompt[:-split_k]; decoder is primed on the last
    split_k tokens, mirroring the training split-1 convention."""
    model.eval()

    items = []
    for key, value in test_data.items():
        text = value["text"] if isinstance(value, dict) else value
        ids = tok.encode(text).ids
        if len(ids) <= split_k:
            raise ValueError(f"prompt {key}: {len(ids)} tokens <= split_k={split_k}")
        items.append(dict(key=str(key), text=text, ids=ids))
    items.sort(key=lambda d: len(d["ids"]))

    real = padded = 0
    results, n_batches = {}, (len(items) + batch_size - 1) // batch_size

    for bi, start in enumerate(range(0, len(items), batch_size), 1):
        batch = items[start:start + batch_size]
        B = len(batch)

        enc_ids = [d["ids"][:-split_k] for d in batch]
        dec_ids = [d["ids"][-split_k:] for d in batch]
        max_enc = max(len(e) for e in enc_ids)

        real   += sum(len(e) for e in enc_ids)
        padded += B * max_enc

        enc = torch.full((B, max_enc), PAD_ID, dtype=torch.long, device=DEVICE)
        for b, e in enumerate(enc_ids):                       # LEFT pad
            enc[b, max_enc - len(e):] = torch.tensor(e, dtype=torch.long, device=DEVICE)

        dec = torch.tensor(dec_ids, dtype=torch.long, device=DEVICE)

        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, cross_kv = encode_prompt(model, enc)
            logits, caches = decoder_forward(model, dec, None, cross_kv, pos0=0)

        history = torch.cat([enc, dec], dim=1)
        ngram_maps = [dict() for _ in range(B)]
        finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        generated = [[] for _ in range(B)]

        for gstep in range(max_new_tokens):
            nxt = sample_next_token(
                logits[:, -1, :], history, ngram_maps,
                temperature=temperature, top_k=top_k, top_p=top_p,
                repetition_penalty=repetition_penalty,
                repetition_window=repetition_window,
                no_repeat_ngram_size=no_repeat_ngram_size)

            nxt = torch.where(finished.unsqueeze(1),
                              torch.full_like(nxt, PAD_ID), nxt)
            history = torch.cat([history, nxt], dim=1)

            if no_repeat_ngram_size:
                update_ngram_maps(ngram_maps, history, no_repeat_ngram_size)

            done = finished.tolist()
            for b, t in enumerate(nxt[:, 0].tolist()):
                if not done[b]:
                    generated[b].append(t)

            if (gstep + 1) % check_every == 0 or gstep + 1 == max_new_tokens:
                for b in range(B):
                    if not finished[b] and len(tok.decode(generated[b]).split()) >= target_words:
                        finished[b] = True
                if bool(finished.all()):
                    break

            if gstep + 1 < max_new_tokens:
                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                    logits, caches = step_b(model, nxt, caches, cross_kv)

        for b, d in enumerate(batch):
            results[d["key"]] = dict(prompt=d["text"], generated_ids=generated[b])

        del caches, cross_kv, history, enc, dec, logits
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        if verbose:
            print(f"  batch {bi}/{n_batches} | enc {max_enc} | "
                  f"gen {max(len(g) for g in generated)} tok", flush=True)

    if verbose:
        print(f"encoder padding waste: {100*(padded-real)/padded:.2f}%")

    return {str(k): results[str(k)] for k in test_data}

In [16]:
import torch
import torch.nn.functional as F

NEG = -1e4          # finite: -inf can make a fully-masked row NaN in fp16

SPECIAL_TOKEN_IDS = [i for i in (
    tok.token_to_id("<|pad|>"), tok.token_to_id("<|unk|>"),
    tok.token_to_id("<|bos|>"), tok.token_to_id("<|eos|>")) if i is not None]
print("banned ids:", SPECIAL_TOKEN_IDS)


@torch.no_grad()
def sample_next_token(logits, history, ngram_maps=None,
                      temperature=1.0, top_k=None, top_p=None,
                      repetition_penalty=1.0, repetition_window=128,
                      no_repeat_ngram_size=None):
    """logits (B, vocab); history (B, T) = prompt + generated so far.
    ngram_maps: list of dicts, one per row, carried across calls."""
    logits = logits.float().clone()
    logits[:, SPECIAL_TOKEN_IDS] = NEG

    # --- repetition penalty over recent history (prompt included) ---
    if repetition_penalty != 1.0 and history.shape[1] > 0:
        recent = history[:, -repetition_window:]
        gathered = torch.gather(logits, 1, recent)
        gathered = torch.where(gathered > 0,
                               gathered / repetition_penalty,
                               gathered * repetition_penalty)
        logits.scatter_(1, recent, gathered)

    # --- no-repeat n-gram, incremental ---
    n = no_repeat_ngram_size or 0
    if n > 1 and ngram_maps is not None:
        for b in range(logits.shape[0]):
            seq = history[b].tolist()
            if len(seq) >= n - 1:
                banned = ngram_maps[b].get(tuple(seq[-(n - 1):]))
                if banned:
                    logits[b, list(banned)] = NEG

    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)

    logits = logits / temperature

    if top_k:
        k = min(top_k, logits.shape[-1])
        kth = logits.topk(k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, NEG)

    if top_p is not None and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cum = sl.softmax(-1).cumsum(-1)
        drop = cum > top_p
        drop[:, 1:] = drop[:, :-1].clone()      # keep the token that crosses p
        drop[:, 0] = False
        logits = logits.masked_fill(
            torch.zeros_like(drop).scatter_(1, si, drop), NEG)

    probs = logits.softmax(-1)
    bad = ~torch.isfinite(probs).all(-1) | (probs.sum(-1) <= 0)
    if bad.any():                                # degenerate row -> greedy
        probs[bad] = 0.0
        probs[bad, logits[bad].argmax(-1)] = 1.0
    return torch.multinomial(probs, 1)


def update_ngram_maps(ngram_maps, history, n):
    """Call after appending each new token. Records the ngram just completed."""
    if n > 1:
        for b in range(history.shape[0]):
            seq = history[b].tolist()
            if len(seq) >= n:
                ngram_maps[b].setdefault(tuple(seq[-n:-1]), set()).add(seq[-1])

banned ids: [0, 1, 2, 3]


In [18]:
def postprocess_predictions(results, test_data, target_words=128):
    preds, short = {}, []

    for key in test_data:
        key = str(key)
        words = tok.decode(results[key]["generated_ids"]).split()
        if len(words) < target_words:
            short.append((key, len(words)))
        preds[key] = {"text": " ".join(words[:target_words])}

    assert set(preds) == {str(k) for k in test_data}, "key set mismatch"

    for key, p in preds.items():
        pt = test_data[key]["text"] if isinstance(test_data[key], dict) else test_data[key]
        assert not p["text"].startswith(pt[:80]), f"echoes prompt: {key}"

    print(f"postprocess | short: {len(short)}/{len(preds)}"
          + (f" -> {short[:5]}" if short else ""))
    return preds, short

In [19]:
five = {str(k): test_data[k] for k in list(test_data)[:5]}

for sk in (16, 32):
    print(f"\n{'='*70}\nsplit_k = {sk}\n{'='*70}")
    r = generate_batched_b(five, model_b, split_k=sk, batch_size=5, verbose=False)
    p, short = postprocess_predictions(r, five)
    print(f"\n{p['1']['text'][:600]}")


split_k = 16
postprocess | short: 0/5

clear from the above that the judgment of the Constitution Bench in [V.S.S.A.C. v. State of Kerala] (1997 (4) SCC 777) was relied upon by the respondent which was relied upon in support of the submission that the decision of the Constitution Bench in the case of [Haikul Nath v. Union of India] (2001 (1) SCC 1), was not correct. 9. The facts in the present case are: (i) that the respondents were the owners of the lands in question in the year 1975. The respondents were not the owners of the lands in question and were the owners of the lands. The respondents filed a writ petition challenging the

split_k = 32
postprocess | short: 0/5

a case where an order granting relief of interim relief was passed against the petitioner by the State Government, which was under challenge before the High Court. In other words, it is difficult to say that the writ petitioners could not have been made parties to the proceedings. If the writ petitioners are aggrieved

In [20]:
!pip -q install rouge-score sacrebleu bert-score

import json, os, zipfile
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import BERTScorer

# BERTScore's roberta-large is an evaluation backbone only — it mirrors the
# official metric and is never part of the trained model or the submission.
SCORER = BERTScorer(model_type="roberta-large", lang="en",
                    rescale_with_baseline=False, device=DEVICE, batch_size=32)
ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

assert set(VAL_PROMPTS) == set(VAL_REFS)
print(f"{len(VAL_PROMPTS)} validation prompts")


def score_config(cfg, prompts=None, refs=None, batch_size=32, max_new_tokens=350):
    prompts = prompts or VAL_PROMPTS
    refs    = refs or VAL_REFS
    print(f"\n{'='*70}\n{cfg['name']}\n{'='*70}", flush=True)

    gen = generate_batched(
        prompts, batch_size=batch_size, max_new_tokens=max_new_tokens,
        target_words=128, check_every=16,
        temperature=cfg["temperature"], top_p=cfg.get("top_p", 0.9),
        top_k=cfg.get("top_k", 50),
        repetition_penalty=cfg.get("repetition_penalty", 1.0),
        repetition_window=cfg.get("repetition_window", 128),
        no_repeat_ngram_size=cfg.get("no_repeat_ngram_size"),
        verbose=False)

    preds_d, short = postprocess_predictions(gen, prompts, target_words=128)

    keys  = list(prompts)
    preds = [preds_d[k]["text"] for k in keys]
    refs_t = [refs[k]["text"] for k in keys]
    srcs   = [refs[k].get("src", "unknown") for k in keys]

    _, _, f1 = SCORER.score(preds, refs_t)
    f1 = f1.cpu().numpy()

    rl = np.array([ROUGE.score(r, p)["rougeL"].fmeasure
                   for p, r in zip(preds, refs_t)])
    bleu = sacrebleu.corpus_bleu(preds, [refs_t]).score / 100.0

    by_src = {s: dict(n=int((np.array(srcs) == s).sum()),
                      f1=float(f1[np.array(srcs) == s].mean()))
              for s in sorted(set(srcs))}

    print(f"  BERTScore F1 {f1.mean():.4f} | ROUGE-L {rl.mean():.4f} | "
          f"BLEU-4 {bleu:.4f} | short {len(short)}/{len(preds)}")
    for s, v in by_src.items():
        print(f"    {s:16s} n={v['n']:3d}  F1={v['f1']:.4f}")

    return dict(name=cfg["name"], bert_f1=float(f1.mean()),
                rouge_l=float(rl.mean()), bleu4=bleu,
                short=len(short), by_src=by_src, cfg=cfg)


CONFIGS = [
    dict(name="t0.7",       temperature=0.7, top_p=0.9, top_k=50, repetition_penalty=1.1),
    dict(name="t0.7_ng4",   temperature=0.7, top_p=0.9, top_k=50, repetition_penalty=1.0,
         no_repeat_ngram_size=4),
    dict(name="t0.7_rp1.2", temperature=0.7, top_p=0.9, top_k=50, repetition_penalty=1.2),
    dict(name="t0.3",       temperature=0.3, top_p=0.9, top_k=50, repetition_penalty=1.1),
    dict(name="greedy_ng3", temperature=0.0, top_p=1.0, top_k=None, repetition_penalty=1.0,
         no_repeat_ngram_size=3),
    dict(name="t0.9",       temperature=0.9, top_p=0.9, top_k=50, repetition_penalty=1.1),
]

results = [score_config(c) for c in CONFIGS]

table = pd.DataFrame([{k: r[k] for k in
                       ("name", "bert_f1", "rouge_l", "bleu4", "short")}
                      for r in results]).sort_values("bert_f1", ascending=False)
print("\n" + "=" * 70 + "\nDECODING SWEEP\n" + "=" * 70)
print(table.to_string(index=False))

best = max(results, key=lambda r: r["bert_f1"])
print(f"\nper-source ({best['name']}):")
print(pd.DataFrame([dict(source=s, n=v["n"], bert_f1=v["f1"])
                    for s, v in best["by_src"].items()]
                   ).sort_values("bert_f1", ascending=False).to_string(index=False))


# ---------- top two -> full test set -> zips ----------
top2 = [r["cfg"] for r in sorted(results, key=lambda r: -r["bert_f1"])[:2]]
print("\ntop two:", [c["name"] for c in top2])

os.makedirs("/kaggle/working/submissions", exist_ok=True)

for cfg in top2:
    print(f"\n--- full test: {cfg['name']} ---", flush=True)
    gen = generate_batched(
        test_data, batch_size=32, max_new_tokens=350, target_words=128,
        temperature=cfg["temperature"], top_p=cfg.get("top_p", 0.9),
        top_k=cfg.get("top_k", 50),
        repetition_penalty=cfg.get("repetition_penalty", 1.0),
        repetition_window=cfg.get("repetition_window", 128),
        no_repeat_ngram_size=cfg.get("no_repeat_ngram_size"),
        verbose=False)

    preds, short = postprocess_predictions(gen, test_data, target_words=128)
    assert set(preds) == {str(k) for k in test_data}, "key mismatch"
    assert all(len(v["text"].split()) <= 128 for v in preds.values()), "over-length"

    jp = f"/kaggle/working/submissions/prediction_{cfg['name']}.json"
    zp = f"/kaggle/working/submissions/A_{cfg['name']}.zip"
    json.dump(preds, open(jp, "w"), ensure_ascii=False)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")

    print(f"  {zp} | {len(short)} short | {os.path.getsize(zp)/1e6:.2f} MB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 3.1 MB/s eta 0:00:00a 0:00:01


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


300 validation prompts

t0.7


NameError: name 'generate_batched' is not defined

In [21]:
for sk in (16, 32):
    r = generate_batched_b(VAL_PROMPTS, model_b, split_k=sk,
                           batch_size=32, max_new_tokens=350, verbose=False)
    p, short = postprocess_predictions(r, VAL_PROMPTS)
    keys = list(VAL_PROMPTS)
    _, _, f1 = SCORER.score([p[k]["text"] for k in keys],
                            [VAL_REFS[k]["text"] for k in keys])
    print(f"split_k={sk}: BERTScore F1 {f1.mean():.4f} | short {len(short)}/300")

postprocess | short: 4/300 -> [('51', 56), ('107', 95), ('163', 88), ('287', 88)]
split_k=16: BERTScore F1 0.8408 | short 4/300
postprocess | short: 2/300 -> [('145', 127), ('185', 91)]
split_k=32: BERTScore F1 0.8417 | short 2/300


In [22]:
import json, zipfile, os

r = generate_batched_b(test_data, model_b, split_k=32,
                       batch_size=32, max_new_tokens=350, verbose=True)
preds, short = postprocess_predictions(r, test_data)

assert set(preds) == {str(k) for k in test_data}, "key mismatch"
assert all(len(v["text"].split()) <= 128 for v in preds.values()), "over-length"

os.makedirs("/kaggle/working/submissions", exist_ok=True)
jp = "/kaggle/working/submissions/prediction.json"
zp = "/kaggle/working/submissions/B_encdec_sk32.zip"

json.dump(preds, open(jp, "w"), ensure_ascii=False)
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(jp, arcname="prediction.json")

with zipfile.ZipFile(zp) as z:
    assert z.namelist() == ["prediction.json"], z.namelist()

print(f"OK — {len(preds)} ids, {len(short)} short, {os.path.getsize(zp)/1e6:.2f} MB")

  batch 1/32 | enc 531 | gen 304 tok
  batch 2/32 | enc 536 | gen 256 tok
  batch 3/32 | enc 542 | gen 224 tok
  batch 4/32 | enc 546 | gen 350 tok
  batch 5/32 | enc 549 | gen 240 tok
  batch 6/32 | enc 553 | gen 336 tok
  batch 7/32 | enc 555 | gen 320 tok
  batch 8/32 | enc 558 | gen 192 tok
  batch 9/32 | enc 560 | gen 256 tok
  batch 10/32 | enc 562 | gen 304 tok
  batch 11/32 | enc 564 | gen 350 tok
  batch 12/32 | enc 566 | gen 224 tok
  batch 13/32 | enc 569 | gen 192 tok
  batch 14/32 | enc 571 | gen 240 tok
  batch 15/32 | enc 574 | gen 288 tok
  batch 16/32 | enc 577 | gen 240 tok
  batch 17/32 | enc 579 | gen 256 tok
  batch 18/32 | enc 582 | gen 240 tok
  batch 19/32 | enc 584 | gen 208 tok
  batch 20/32 | enc 588 | gen 256 tok
  batch 21/32 | enc 591 | gen 192 tok
  batch 22/32 | enc 593 | gen 256 tok
  batch 23/32 | enc 597 | gen 208 tok
  batch 24/32 | enc 601 | gen 272 tok
  batch 25/32 | enc 607 | gen 224 tok
  batch 26/32 | enc 611 | gen 272 tok
  batch 27/32 | enc 6